# 1. Temporal sensitivity prerequisites audit

This notebook prepares the temporal-leakage sensitivity experiment without rebuilding any model inputs yet.

Its purpose is to establish the exact reconstruction contract:

- authoritative baseline-date field;
- selected measurement-date field in every baseline-aligned manifest;
- corresponding measurement-date field in every cleaned longitudinal source;
- existing signed and absolute baseline offsets;
- first AD/conversion date available for pMCI participants;
- tie situations in the original nearest-measurement selection;
- source-level validity and QC fields that must remain respected;
- whether an earlier eligible MRI can be reselected.

The notebook uses the exact project paths already used by the multimodal pipeline. It saves all findings under a dedicated audit folder and does not modify the existing manifests or model-ready inputs.

## 1.1. Mount Drive and define the exact project files

In [ ]:
# ============================================================
# 1. Mount Drive and define the exact project files
# ============================================================

from pathlib import Path
import json
import re

import numpy as np
import pandas as pd

from IPython.display import display


try:
    from google.colab import drive

    if not Path(
        "/content/drive/MyDrive"
    ).exists():
        drive.mount(
            "/content/drive"
        )

except ImportError:
    pass


DRIVE_ROOT = Path(
    "/content/drive/MyDrive"
)

PROJECT_ROOT = (
    DRIVE_ROOT
    / "adni_mri"
)

NON_IMAGING_ROOT = (
    PROJECT_ROOT
    / "adni_non_imaging"
)

MODEL_ROOT = (
    PROJECT_ROOT
    / "models"
    / "3mt_tmc_evidential"
)


AUTHORITATIVE_COHORT_DIR = (
    NON_IMAGING_ROOT
    / "manifests"
    / "authoritative_clinical_cohort"
)

COHORT_PATH = (
    AUTHORITATIVE_COHORT_DIR
    / "authoritative_four_group_clinical_cohort.csv"
)

MCI_TRAJECTORY_PATH = (
    AUTHORITATIVE_COHORT_DIR
    / "authoritative_mci_36m_trajectory_labels.csv"
)

MCI_EXCLUSIONS_PATH = (
    AUTHORITATIVE_COHORT_DIR
    / "authoritative_mci_trajectory_exclusions.csv"
)


BASELINE_ALIGNED_ROOT = (
    NON_IMAGING_ROOT
    / "manifests"
    / "baseline_aligned_modalities"
)


MANIFEST_PATHS = {
    "ptdemog": (
        BASELINE_ALIGNED_ROOT
        / "ptdemog"
        / "ptdemog_baseline_aligned_cohort.csv"
    ),

    "adas": (
        BASELINE_ALIGNED_ROOT
        / "adas"
        / "adas_baseline_aligned_cohort.csv"
    ),

    "mmse": (
        BASELINE_ALIGNED_ROOT
        / "mmse"
        / "mmse_baseline_aligned_cohort.csv"
    ),

    "faq": (
        BASELINE_ALIGNED_ROOT
        / "faq"
        / "faq_baseline_aligned_cohort.csv"
    ),

    "csf": (
        BASELINE_ALIGNED_ROOT
        / "csf_core_biomarkers"
        / "csf_core_biomarkers_baseline_aligned_cohort.csv"
    ),

    "plasma": (
        BASELINE_ALIGNED_ROOT
        / "plasma"
        / "plasma_baseline_aligned_cohort.csv"
    ),

    "apoe": (
        BASELINE_ALIGNED_ROOT
        / "apoe"
        / "apoe_baseline_aligned_cohort.csv"
    ),

    "mri": (
        BASELINE_ALIGNED_ROOT
        / "mri"
        / "mri_baseline_aligned_authoritative_cohort_1057.csv"
    ),
}


SOURCE_PATHS = {
    "ptdemog": (
        NON_IMAGING_ROOT
        / "interim"
        / "ptdemog_cleaned_longitudinal.csv"
    ),

    "adas": (
        NON_IMAGING_ROOT
        / "processed"
        / "adas_clean_longitudinal_interim.csv"
    ),

    "mmse": (
        NON_IMAGING_ROOT
        / "processed"
        / "mmse"
        / "mmse_longitudinal_primary_features_v2.csv"
    ),

    "faq": (
        NON_IMAGING_ROOT
        / "processed"
        / "faq_model_ready_dated_longitudinal.csv"
    ),

    "csf": (
        NON_IMAGING_ROOT
        / "interim"
        / "csf_core_biomarkers"
        / "csf_core_biomarkers_cleaned_visit_level.csv"
    ),

    "plasma": (
        NON_IMAGING_ROOT
        / "processed"
        / "plasma"
        / "plasma_biomarkers_cleaned_longitudinal.csv"
    ),

    "apoe": (
        NON_IMAGING_ROOT
        / "processed"
        / "apoe"
        / "apoe_genotype_cleaned_participant_level.csv"
    ),
}


MRI_PROCESSED_ROOT = (
    PROJECT_ROOT
    / "processed"
    / "mni_n4_syn_rectcrop_177x213x183_t1norm_p01p99"
)

MRI_NPY_DIR = (
    MRI_PROCESSED_ROOT
    / "npy_t1"
)

MRI_NII_DIR = (
    MRI_PROCESSED_ROOT
    / "nii_t1"
)


AUDIT_ROOT = (
    MODEL_ROOT
    / "temporal_leakage_prerequisites_audit"
)

TABLE_DIR = (
    AUDIT_ROOT
    / "tables"
)

CONTRACT_DIR = (
    AUDIT_ROOT
    / "contract"
)

for directory in [
    AUDIT_ROOT,
    TABLE_DIR,
    CONTRACT_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


SYMMETRIC_WINDOWS_DAYS = {
    "adas": 90,
    "mmse": 90,
    "faq": 90,
    "mri": 90,
    "csf": 180,
    "plasma": 180,
}


print("=" * 72)
print("TEMPORAL-LEAKAGE RECONSTRUCTION PREREQUISITES AUDIT")
print("=" * 72)

print(
    f"\nAuthoritative cohort:\n"
    f"{COHORT_PATH}"
)

print(
    f"\nAudit outputs:\n"
    f"{AUDIT_ROOT}"
)

## 1.2. Confirm the exact files used by the existing pipeline

This is a one-time inventory. Missing files are shown explicitly before any content is inspected.

In [ ]:
# ============================================================
# 2. Confirm the exact files used by the existing pipeline
# ============================================================

required_file_rows = [
    {
        "ROLE": "authoritative_cohort",
        "MODALITY": "cohort",
        "PATH": str(COHORT_PATH),
        "EXISTS": COHORT_PATH.is_file(),
    },
    {
        "ROLE": "mci_trajectory",
        "MODALITY": "trajectory",
        "PATH": str(MCI_TRAJECTORY_PATH),
        "EXISTS": MCI_TRAJECTORY_PATH.is_file(),
    },
    {
        "ROLE": "mci_exclusions",
        "MODALITY": "trajectory",
        "PATH": str(MCI_EXCLUSIONS_PATH),
        "EXISTS": MCI_EXCLUSIONS_PATH.is_file(),
    },
]

for modality, path in MANIFEST_PATHS.items():
    required_file_rows.append(
        {
            "ROLE": "baseline_aligned_manifest",
            "MODALITY": modality,
            "PATH": str(path),
            "EXISTS": path.is_file(),
        }
    )

for modality, path in SOURCE_PATHS.items():
    required_file_rows.append(
        {
            "ROLE": "cleaned_source",
            "MODALITY": modality,
            "PATH": str(path),
            "EXISTS": path.is_file(),
        }
    )


required_file_inventory = pd.DataFrame(
    required_file_rows
)


REQUIRED_FILE_INVENTORY_PATH = (
    TABLE_DIR
    / "required_file_inventory.csv"
)

required_file_inventory.to_csv(
    REQUIRED_FILE_INVENTORY_PATH,
    index=False,
)


display(
    required_file_inventory
)


missing_required_files = (
    required_file_inventory.loc[
        ~required_file_inventory[
            "EXISTS"
        ]
    ]
)


if not missing_required_files.empty:
    raise FileNotFoundError(
        "The exact pipeline files below are missing:\n\n"
        + missing_required_files[
            [
                "ROLE",
                "MODALITY",
                "PATH",
            ]
        ].to_string(
            index=False
        )
    )


print(
    "\nAll exact cohort, trajectory, manifest, "
    "and cleaned-source files were found."
)

## 1.3. Load the cohort, trajectory files, manifests, and cleaned sources

Loading is conservative. Column names are stripped and obvious date columns are parsed, but no rows are filtered and no values are imputed.

In [ ]:
# ============================================================
# 3. Load all prerequisite tables
# ============================================================

DATE_NAME_PATTERN = re.compile(
    r"(DATE|VISDATE|EXAMDATE|TESTDT|APTESTDT|SCANDATE)",
    flags=re.IGNORECASE,
)


def load_adni_table(
    path,
    source_name,
):
    table = pd.read_csv(
        path,
        low_memory=False,
    )

    table.columns = (
        table.columns
        .astype(str)
        .str.strip()
    )

    if "RID" in table.columns:
        table["RID"] = (
            pd.to_numeric(
                table["RID"],
                errors="coerce",
            )
            .astype("Int64")
        )

    for column in table.columns:
        if DATE_NAME_PATTERN.search(
            str(column)
        ):
            parsed = pd.to_datetime(
                table[column],
                errors="coerce",
            )

            if parsed.notna().any():
                table[column] = parsed

    print(
        f"{source_name:<42} "
        f"rows={len(table):>7,} | "
        f"columns={table.shape[1]:>3} | "
        f"unique RID="
        f"{table['RID'].nunique(dropna=True) if 'RID' in table.columns else 'absent'}"
    )

    return table


cohort_df = load_adni_table(
    COHORT_PATH,
    "authoritative cohort",
)

trajectory_df = load_adni_table(
    MCI_TRAJECTORY_PATH,
    "MCI trajectory labels",
)

exclusions_df = load_adni_table(
    MCI_EXCLUSIONS_PATH,
    "MCI trajectory exclusions",
)


manifest_dfs = {
    modality: load_adni_table(
        path,
        f"{modality} manifest",
    )
    for modality, path in MANIFEST_PATHS.items()
}


source_dfs = {
    modality: load_adni_table(
        path,
        f"{modality} source",
    )
    for modality, path in SOURCE_PATHS.items()
}

## 1.4. Save complete column inventories

These tables remove any uncertainty about the exact field names available in each source.

In [ ]:
# ============================================================
# 4. Complete column inventories
# ============================================================

def column_inventory(
    table,
    source_role,
    source_name,
):
    rows = []

    for position, column in enumerate(
        table.columns
    ):
        series = table[column]

        rows.append(
            {
                "SOURCE_ROLE": source_role,
                "SOURCE_NAME": source_name,
                "COLUMN_POSITION": position,
                "COLUMN": column,
                "DTYPE": str(series.dtype),
                "NON_MISSING": int(
                    series.notna().sum()
                ),
                "UNIQUE_NON_MISSING": int(
                    series.nunique(
                        dropna=True
                    )
                ),
                "IS_PARSED_DATETIME": bool(
                    pd.api.types.is_datetime64_any_dtype(
                        series
                    )
                ),
            }
        )

    return rows


inventory_rows = []

inventory_rows.extend(
    column_inventory(
        cohort_df,
        "authoritative_cohort",
        "cohort",
    )
)

inventory_rows.extend(
    column_inventory(
        trajectory_df,
        "trajectory",
        "mci_trajectory",
    )
)

inventory_rows.extend(
    column_inventory(
        exclusions_df,
        "trajectory",
        "mci_exclusions",
    )
)

for modality, table in manifest_dfs.items():
    inventory_rows.extend(
        column_inventory(
            table,
            "baseline_aligned_manifest",
            modality,
        )
    )

for modality, table in source_dfs.items():
    inventory_rows.extend(
        column_inventory(
            table,
            "cleaned_source",
            modality,
        )
    )


complete_column_inventory = pd.DataFrame(
    inventory_rows
)


COMPLETE_COLUMN_INVENTORY_PATH = (
    TABLE_DIR
    / "complete_column_inventory.csv"
)

complete_column_inventory.to_csv(
    COMPLETE_COLUMN_INVENTORY_PATH,
    index=False,
)


print(
    f"Complete inventory saved to:\n"
    f"{COMPLETE_COLUMN_INVENTORY_PATH}"
)

display(
    complete_column_inventory
)

## 1.5. Identify all date and baseline-offset fields

This section narrows the complete inventory to fields that can define:

- authoritative baseline;
- selected modality date;
- longitudinal source date;
- first AD or conversion date;
- signed and absolute distance from baseline.

In [ ]:
# ============================================================
# 5. Date and timing field inventory
# ============================================================

DATE_OR_TIMING_PATTERN = re.compile(
    (
        r"DATE|VISDATE|EXAMDATE|TESTDT|APTESTDT|SCANDATE|"
        r"DAYS_FROM_BASELINE|ABS_DAYS_FROM_BASELINE|"
        r"CONVERSION|FIRST_AD|AD_DATE|BASELINE"
    ),
    flags=re.IGNORECASE,
)


date_and_timing_inventory = (
    complete_column_inventory.loc[
        complete_column_inventory[
            "COLUMN"
        ].str.contains(
            DATE_OR_TIMING_PATTERN,
            na=False,
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


DATE_AND_TIMING_INVENTORY_PATH = (
    TABLE_DIR
    / "date_and_timing_column_inventory.csv"
)

date_and_timing_inventory.to_csv(
    DATE_AND_TIMING_INVENTORY_PATH,
    index=False,
)


print("=" * 72)
print("DATE AND TIMING FIELDS")
print("=" * 72)

display(
    date_and_timing_inventory
)

print(
    f"\nSaved to:\n"
    f"{DATE_AND_TIMING_INVENTORY_PATH}"
)

## 1.6. Identify likely authoritative baseline and pMCI conversion fields

The output separates candidates by meaning. No candidate is silently selected when several plausible columns exist.

In [ ]:
# ============================================================
# 6. Cohort and trajectory date candidates
# ============================================================

def matching_columns(
    table,
    patterns,
):
    compiled_patterns = [
        re.compile(
            pattern,
            flags=re.IGNORECASE,
        )
        for pattern in patterns
    ]

    return [
        column
        for column in table.columns
        if any(
            pattern.search(
                str(column)
            )
            for pattern in compiled_patterns
        )
    ]


cohort_baseline_candidates = matching_columns(
    cohort_df,
    [
        r"BASELINE.*DATE",
        r"DATE.*BASELINE",
        r"BL.*DATE",
    ],
)

cohort_conversion_candidates = matching_columns(
    cohort_df,
    [
        r"FIRST.*AD",
        r"AD.*FIRST",
        r"CONVERSION.*DATE",
        r"DATE.*CONVERSION",
        r"AD.*DATE",
    ],
)

trajectory_baseline_candidates = matching_columns(
    trajectory_df,
    [
        r"BASELINE.*DATE",
        r"DATE.*BASELINE",
        r"BL.*DATE",
    ],
)

trajectory_conversion_candidates = matching_columns(
    trajectory_df,
    [
        r"FIRST.*AD",
        r"AD.*FIRST",
        r"CONVERSION.*DATE",
        r"DATE.*CONVERSION",
        r"AD.*DATE",
    ],
)


cohort_and_trajectory_candidates = pd.DataFrame(
    [
        {
            "TABLE": "authoritative_cohort",
            "PURPOSE": "baseline_date",
            "CANDIDATES": json.dumps(
                cohort_baseline_candidates
            ),
            "CANDIDATE_COUNT": len(
                cohort_baseline_candidates
            ),
        },
        {
            "TABLE": "authoritative_cohort",
            "PURPOSE": "first_ad_or_conversion_date",
            "CANDIDATES": json.dumps(
                cohort_conversion_candidates
            ),
            "CANDIDATE_COUNT": len(
                cohort_conversion_candidates
            ),
        },
        {
            "TABLE": "mci_trajectory",
            "PURPOSE": "baseline_date",
            "CANDIDATES": json.dumps(
                trajectory_baseline_candidates
            ),
            "CANDIDATE_COUNT": len(
                trajectory_baseline_candidates
            ),
        },
        {
            "TABLE": "mci_trajectory",
            "PURPOSE": "first_ad_or_conversion_date",
            "CANDIDATES": json.dumps(
                trajectory_conversion_candidates
            ),
            "CANDIDATE_COUNT": len(
                trajectory_conversion_candidates
            ),
        },
    ]
)


COHORT_TRAJECTORY_CANDIDATES_PATH = (
    TABLE_DIR
    / "cohort_and_trajectory_date_candidates.csv"
)

cohort_and_trajectory_candidates.to_csv(
    COHORT_TRAJECTORY_CANDIDATES_PATH,
    index=False,
)


display(
    cohort_and_trajectory_candidates
)


print(
    "\nAuthoritative cohort columns:"
)

print(
    cohort_df.columns.tolist()
)


print(
    "\nMCI trajectory columns:"
)

print(
    trajectory_df.columns.tolist()
)

## 1.7. Identify selected-date and longitudinal-date candidates by modality

The reconstruction notebook will later use the resolved date fields from this table directly.

In [ ]:
# ============================================================
# 7. Modality date candidates
# ============================================================

modality_date_rows = []


for modality in [
    "adas",
    "mmse",
    "faq",
    "csf",
    "plasma",
    "mri",
]:

    manifest_table = manifest_dfs[
        modality
    ]

    manifest_date_candidates = [
        column
        for column in manifest_table.columns
        if (
            DATE_NAME_PATTERN.search(
                str(column)
            )
            or
            "DAYS_FROM_BASELINE"
            in str(column).upper()
        )
    ]


    if modality == "mri":
        source_date_candidates = []

    else:
        source_table = source_dfs[
            modality
        ]

        source_date_candidates = [
            column
            for column in source_table.columns
            if DATE_NAME_PATTERN.search(
                str(column)
            )
        ]


    signed_offset_candidates = [
        column
        for column in manifest_table.columns
        if (
            "DAYS_FROM_BASELINE"
            in str(column).upper()
            and
            "ABS"
            not in str(column).upper()
        )
    ]

    absolute_offset_candidates = [
        column
        for column in manifest_table.columns
        if (
            "ABS_DAYS_FROM_BASELINE"
            in str(column).upper()
        )
    ]


    modality_date_rows.append(
        {
            "MODALITY": modality,
            "WINDOW_DAYS": SYMMETRIC_WINDOWS_DAYS[
                modality
            ],
            "MANIFEST_DATE_CANDIDATES": json.dumps(
                manifest_date_candidates
            ),
            "MANIFEST_DATE_CANDIDATE_COUNT": len(
                manifest_date_candidates
            ),
            "SOURCE_DATE_CANDIDATES": json.dumps(
                source_date_candidates
            ),
            "SOURCE_DATE_CANDIDATE_COUNT": len(
                source_date_candidates
            ),
            "SIGNED_OFFSET_CANDIDATES": json.dumps(
                signed_offset_candidates
            ),
            "SIGNED_OFFSET_CANDIDATE_COUNT": len(
                signed_offset_candidates
            ),
            "ABSOLUTE_OFFSET_CANDIDATES": json.dumps(
                absolute_offset_candidates
            ),
            "ABSOLUTE_OFFSET_CANDIDATE_COUNT": len(
                absolute_offset_candidates
            ),
        }
    )


modality_date_candidates = pd.DataFrame(
    modality_date_rows
)


MODALITY_DATE_CANDIDATES_PATH = (
    TABLE_DIR
    / "modality_date_candidates.csv"
)

modality_date_candidates.to_csv(
    MODALITY_DATE_CANDIDATES_PATH,
    index=False,
)


display(
    modality_date_candidates
)


for modality in [
    "adas",
    "mmse",
    "faq",
    "csf",
    "plasma",
    "mri",
]:
    print(
        "\n" + "=" * 72
    )

    print(
        modality.upper()
    )

    print(
        "=" * 72
    )

    print(
        "\nManifest columns:"
    )

    print(
        manifest_dfs[
            modality
        ].columns.tolist()
    )

    if modality != "mri":
        print(
            "\nCleaned source columns:"
        )

        print(
            source_dfs[
                modality
            ].columns.tolist()
        )

## 1.8. Inventory source validity, QC, visit, phase, and record-key fields

These fields may have governed which longitudinal rows were eligible. They must be known before pre-baseline reselection.

In [ ]:
# ============================================================
# 8. Source validity and QC field inventory
# ============================================================

QC_FIELD_PATTERN = re.compile(
    (
        r"QC|VALID|USABLE|EXCLUDE|INCLUDE|OUTLIER|CONFLICT|"
        r"MISMATCH|FLAG|COMMENT|STATUS|AVAILABLE|"
        r"VISCODE|PHASE|EXAMDATE|DATE|RID|PTID|ID$|KEY"
    ),
    flags=re.IGNORECASE,
)


source_qc_rows = []


for modality, table in source_dfs.items():

    for column in table.columns:

        if QC_FIELD_PATTERN.search(
            str(column)
        ):
            source_qc_rows.append(
                {
                    "MODALITY": modality,
                    "COLUMN": column,
                    "DTYPE": str(
                        table[column].dtype
                    ),
                    "NON_MISSING": int(
                        table[column].notna().sum()
                    ),
                    "UNIQUE_NON_MISSING": int(
                        table[column].nunique(
                            dropna=True
                        )
                    ),
                }
            )


source_qc_inventory = pd.DataFrame(
    source_qc_rows
)


SOURCE_QC_INVENTORY_PATH = (
    TABLE_DIR
    / "source_qc_and_validity_field_inventory.csv"
)

source_qc_inventory.to_csv(
    SOURCE_QC_INVENTORY_PATH,
    index=False,
)


display(
    source_qc_inventory
)


print(
    f"\nSaved to:\n"
    f"{SOURCE_QC_INVENTORY_PATH}"
)

## 1.9. Profile the existing signed baseline offsets

Where a signed offset already exists, this section reports the exact distribution and the number of selected post-baseline measurements. No reselection occurs.

In [ ]:
# ============================================================
# 9. Existing signed-offset profile
# ============================================================

signed_offset_profile_rows = []


for modality in [
    "adas",
    "mmse",
    "faq",
    "csf",
    "plasma",
    "mri",
]:

    table = manifest_dfs[
        modality
    ]

    signed_candidates = [
        column
        for column in table.columns
        if (
            "DAYS_FROM_BASELINE"
            in str(column).upper()
            and
            "ABS"
            not in str(column).upper()
        )
    ]


    for offset_column in signed_candidates:

        offset_values = pd.to_numeric(
            table[offset_column],
            errors="coerce",
        )

        observed = offset_values.dropna()


        signed_offset_profile_rows.append(
            {
                "MODALITY": modality,
                "OFFSET_COLUMN": offset_column,
                "OBSERVED_OFFSETS": int(
                    observed.size
                ),
                "MINIMUM_DAYS": (
                    float(
                        observed.min()
                    )
                    if not observed.empty
                    else np.nan
                ),
                "MEDIAN_DAYS": (
                    float(
                        observed.median()
                    )
                    if not observed.empty
                    else np.nan
                ),
                "MAXIMUM_DAYS": (
                    float(
                        observed.max()
                    )
                    if not observed.empty
                    else np.nan
                ),
                "BEFORE_BASELINE": int(
                    (observed < 0).sum()
                ),
                "SAME_DAY": int(
                    (observed == 0).sum()
                ),
                "AFTER_BASELINE": int(
                    (observed > 0).sum()
                ),
                "AFTER_1_TO_30_DAYS": int(
                    (
                        (observed >= 1)
                        &
                        (observed <= 30)
                    ).sum()
                ),
                "AFTER_31_TO_90_DAYS": int(
                    (
                        (observed >= 31)
                        &
                        (observed <= 90)
                    ).sum()
                ),
                "AFTER_91_TO_180_DAYS": int(
                    (
                        (observed >= 91)
                        &
                        (observed <= 180)
                    ).sum()
                ),
            }
        )


signed_offset_profile = pd.DataFrame(
    signed_offset_profile_rows
)


SIGNED_OFFSET_PROFILE_PATH = (
    TABLE_DIR
    / "existing_signed_offset_profile.csv"
)

signed_offset_profile.to_csv(
    SIGNED_OFFSET_PROFILE_PATH,
    index=False,
)


display(
    signed_offset_profile
)


print(
    f"\nSaved to:\n"
    f"{SIGNED_OFFSET_PROFILE_PATH}"
)

## 1.10. Determine whether first AD dates can be attached to all pMCI participants

The cell reports coverage for every plausible conversion-date field. It does not select a field when the meaning is ambiguous.

In [ ]:
# ============================================================
# 10. pMCI first-AD/conversion-date coverage
# ============================================================

cohort_label_candidates = [
    column
    for column in cohort_df.columns
    if str(column).upper()
    in {
        "COHORT_LABEL",
        "CLINICAL_GROUP",
    }
]


if len(
    cohort_label_candidates
) != 1:
    raise ValueError(
        "Expected exactly one authoritative cohort label column "
        "among COHORT_LABEL and CLINICAL_GROUP."
    )


cohort_label_column = cohort_label_candidates[
    0
]


pmci_rids = set(
    cohort_df.loc[
        cohort_df[
            cohort_label_column
        ]
        .astype(str)
        .str.strip()
        .eq(
            "pMCI"
        ),
        "RID",
    ]
    .dropna()
    .astype(int)
    .tolist()
)


conversion_coverage_rows = []


candidate_tables = {
    "authoritative_cohort": cohort_df,
    "mci_trajectory": trajectory_df,
}


for table_name, table in candidate_tables.items():

    conversion_candidates = matching_columns(
        table,
        [
            r"FIRST.*AD",
            r"AD.*FIRST",
            r"CONVERSION.*DATE",
            r"DATE.*CONVERSION",
            r"AD.*DATE",
        ],
    )


    for column in conversion_candidates:

        parsed_dates = pd.to_datetime(
            table[column],
            errors="coerce",
        )

        candidate_rids_with_date = set(
            table.loc[
                parsed_dates.notna(),
                "RID",
            ]
            .dropna()
            .astype(int)
            .tolist()
        )


        conversion_coverage_rows.append(
            {
                "TABLE": table_name,
                "COLUMN": column,
                "pMCI_PARTICIPANTS": len(
                    pmci_rids
                ),
                "pMCI_WITH_NON_MISSING_DATE": len(
                    pmci_rids
                    &
                    candidate_rids_with_date
                ),
                "pMCI_WITHOUT_DATE": len(
                    pmci_rids
                    -
                    candidate_rids_with_date
                ),
            }
        )


conversion_date_coverage = pd.DataFrame(
    conversion_coverage_rows
)


CONVERSION_DATE_COVERAGE_PATH = (
    TABLE_DIR
    / "pmci_conversion_date_coverage.csv"
)

conversion_date_coverage.to_csv(
    CONVERSION_DATE_COVERAGE_PATH,
    index=False,
)


print(
    f"Authoritative pMCI participants: "
    f"{len(pmci_rids)}"
)

display(
    conversion_date_coverage
)


print(
    f"\nSaved to:\n"
    f"{CONVERSION_DATE_COVERAGE_PATH}"
)

## 1.11. Check whether MRI reselection is possible from the available files

This section distinguishes three situations:

1. the MRI manifest already contains enough timing and image identifiers to identify a pre-baseline alternative;
2. the processed image folders contain additional files but no date mapping;
3. another MRI inventory or metadata table is required.

In [ ]:
# ============================================================
# 11. MRI reselection feasibility
# ============================================================

mri_manifest_df = manifest_dfs[
    "mri"
]


mri_identifier_candidates = [
    column
    for column in mri_manifest_df.columns
    if any(
        token in str(column).upper()
        for token in [
            "IMAGE",
            "SERIES",
            "STUDY",
            "VISCODE",
            "PHASE",
            "FILE",
            "PATH",
            "NPY",
            "NII",
        ]
    )
]


mri_date_candidates = [
    column
    for column in mri_manifest_df.columns
    if DATE_NAME_PATTERN.search(
        str(column)
    )
]


mri_offset_candidates = [
    column
    for column in mri_manifest_df.columns
    if "DAYS_FROM_BASELINE"
    in str(column).upper()
]


processed_npy_count = len(
    list(
        MRI_NPY_DIR.glob(
            "*.npy"
        )
    )
)

processed_nii_count = len(
    list(
        MRI_NII_DIR.glob(
            "*.nii*"
        )
    )
)


mri_reselection_feasibility = pd.DataFrame(
    [
        {
            "MRI_MANIFEST_ROWS": len(
                mri_manifest_df
            ),
            "MRI_MANIFEST_UNIQUE_RID": int(
                mri_manifest_df[
                    "RID"
                ].nunique(
                    dropna=True
                )
            ),
            "MRI_DATE_CANDIDATES": json.dumps(
                mri_date_candidates
            ),
            "MRI_OFFSET_CANDIDATES": json.dumps(
                mri_offset_candidates
            ),
            "MRI_IDENTIFIER_OR_PATH_CANDIDATES": json.dumps(
                mri_identifier_candidates
            ),
            "PROCESSED_NPY_FILE_COUNT": processed_npy_count,
            "PROCESSED_NIFTI_FILE_COUNT": processed_nii_count,
            "MANIFEST_HAS_MULTIPLE_ROWS_PER_RID": bool(
                mri_manifest_df[
                    "RID"
                ].duplicated(
                    keep=False
                ).any()
            ),
        }
    ]
)


MRI_RESELECTION_FEASIBILITY_PATH = (
    TABLE_DIR
    / "mri_reselection_feasibility.csv"
)

mri_reselection_feasibility.to_csv(
    MRI_RESELECTION_FEASIBILITY_PATH,
    index=False,
)


display(
    mri_reselection_feasibility.T
)


print(
    "\nMRI manifest columns:"
)

print(
    mri_manifest_df.columns.tolist()
)


print(
    f"\nSaved to:\n"
    f"{MRI_RESELECTION_FEASIBILITY_PATH}"
)

## 1.12. Inspect possible original tie-breaking evidence

For each non-imaging modality, this section reports:

- whether the selected manifest preserves a source row key;
- whether visit code and phase are preserved;
- whether the cleaned source contains duplicate RID-date observations;
- how often more than one source row exists on the same RID and date.

These results determine whether the original selected record can be matched exactly and whether a deterministic tie rule must be reconstructed.

In [ ]:
# ============================================================
# 12. Tie-breaking and exact-record-match prerequisites
# ============================================================

tie_breaking_rows = []


RECORD_KEY_PATTERN = re.compile(
    r"(^|_)(ID|KEY|ROW|REC|RECORD|IMAGE|SERIES|STUDY)($|_)",
    flags=re.IGNORECASE,
)


for modality in [
    "adas",
    "mmse",
    "faq",
    "csf",
    "plasma",
]:

    manifest_table = manifest_dfs[
        modality
    ]

    source_table = source_dfs[
        modality
    ]


    manifest_record_keys = [
        column
        for column in manifest_table.columns
        if RECORD_KEY_PATTERN.search(
            str(column)
        )
    ]

    source_record_keys = [
        column
        for column in source_table.columns
        if RECORD_KEY_PATTERN.search(
            str(column)
        )
    ]

    manifest_visit_fields = [
        column
        for column in manifest_table.columns
        if str(column).upper()
        in {
            "VISCODE",
            "VISCODE2",
            "PHASE",
        }
    ]

    source_visit_fields = [
        column
        for column in source_table.columns
        if str(column).upper()
        in {
            "VISCODE",
            "VISCODE2",
            "PHASE",
        }
    ]


    source_date_candidates = [
        column
        for column in source_table.columns
        if DATE_NAME_PATTERN.search(
            str(column)
        )
    ]


    for date_column in source_date_candidates:

        parsed_date = pd.to_datetime(
            source_table[
                date_column
            ],
            errors="coerce",
        )

        duplicate_rid_date_rows = int(
            pd.DataFrame(
                {
                    "RID": source_table[
                        "RID"
                    ],
                    "DATE": parsed_date,
                }
            )
            .dropna(
                subset=[
                    "RID",
                    "DATE",
                ]
            )
            .duplicated(
                subset=[
                    "RID",
                    "DATE",
                ],
                keep=False,
            )
            .sum()
        )


        tie_breaking_rows.append(
            {
                "MODALITY": modality,
                "SOURCE_DATE_COLUMN": date_column,
                "DUPLICATE_RID_DATE_ROWS": duplicate_rid_date_rows,
                "MANIFEST_RECORD_KEYS": json.dumps(
                    manifest_record_keys
                ),
                "SOURCE_RECORD_KEYS": json.dumps(
                    source_record_keys
                ),
                "MANIFEST_VISIT_FIELDS": json.dumps(
                    manifest_visit_fields
                ),
                "SOURCE_VISIT_FIELDS": json.dumps(
                    source_visit_fields
                ),
            }
        )


tie_breaking_prerequisites = pd.DataFrame(
    tie_breaking_rows
)


TIE_BREAKING_PATH = (
    TABLE_DIR
    / "tie_breaking_and_record_match_prerequisites.csv"
)

tie_breaking_prerequisites.to_csv(
    TIE_BREAKING_PATH,
    index=False,
)


display(
    tie_breaking_prerequisites
)


print(
    f"\nSaved to:\n"
    f"{TIE_BREAKING_PATH}"
)

## 1.13. Produce the reconstruction-readiness summary

A component is marked **automatically resolved** only when exactly one plausible field was found. Ambiguous cases remain explicit rather than being guessed.

In [ ]:
# ============================================================
# 13. Reconstruction-readiness summary
# ============================================================

readiness_rows = []


baseline_candidate_union = list(
    dict.fromkeys(
        cohort_baseline_candidates
        +
        trajectory_baseline_candidates
    )
)

conversion_candidate_union = list(
    dict.fromkeys(
        cohort_conversion_candidates
        +
        trajectory_conversion_candidates
    )
)


readiness_rows.append(
    {
        "COMPONENT": "authoritative_baseline_date",
        "AUTOMATICALLY_RESOLVED": (
            len(
                baseline_candidate_union
            )
            == 1
        ),
        "CANDIDATES": json.dumps(
            baseline_candidate_union
        ),
        "NOTES": (
            "Exactly one candidate is required before rebuilding "
            "signed modality windows."
        ),
    }
)


readiness_rows.append(
    {
        "COMPONENT": "pMCI_first_AD_or_conversion_date",
        "AUTOMATICALLY_RESOLVED": (
            len(
                conversion_candidate_union
            )
            == 1
        ),
        "CANDIDATES": json.dumps(
            conversion_candidate_union
        ),
        "NOTES": (
            "The selected field must cover all pMCI participants "
            "used in the prognosis cohort."
        ),
    }
)


for _, row in modality_date_candidates.iterrows():

    modality = row[
        "MODALITY"
    ]

    manifest_date_candidates = json.loads(
        row[
            "MANIFEST_DATE_CANDIDATES"
        ]
    )

    source_date_candidates = json.loads(
        row[
            "SOURCE_DATE_CANDIDATES"
        ]
    )

    readiness_rows.append(
        {
            "COMPONENT": (
                f"{modality}_selected_manifest_date"
            ),
            "AUTOMATICALLY_RESOLVED": (
                len(
                    manifest_date_candidates
                )
                == 1
            ),
            "CANDIDATES": json.dumps(
                manifest_date_candidates
            ),
            "NOTES": (
                "Date of the observation selected in the existing "
                "baseline-aligned manifest."
            ),
        }
    )

    if modality != "mri":
        readiness_rows.append(
            {
                "COMPONENT": (
                    f"{modality}_longitudinal_source_date"
                ),
                "AUTOMATICALLY_RESOLVED": (
                    len(
                        source_date_candidates
                    )
                    == 1
                ),
                "CANDIDATES": json.dumps(
                    source_date_candidates
                ),
                "NOTES": (
                    "Date used to select the nearest eligible "
                    "pre-baseline source row."
                ),
            }
        )


reconstruction_readiness_summary = pd.DataFrame(
    readiness_rows
)


RECONSTRUCTION_READINESS_PATH = (
    TABLE_DIR
    / "reconstruction_readiness_summary.csv"
)

reconstruction_readiness_summary.to_csv(
    RECONSTRUCTION_READINESS_PATH,
    index=False,
)


display(
    reconstruction_readiness_summary
)


unresolved_components = (
    reconstruction_readiness_summary.loc[
        ~reconstruction_readiness_summary[
            "AUTOMATICALLY_RESOLVED"
        ]
    ]
)


print(
    f"\nAutomatically resolved components: "
    f"{int(reconstruction_readiness_summary['AUTOMATICALLY_RESOLVED'].sum())}"
)

print(
    f"Unresolved or ambiguous components: "
    f"{len(unresolved_components)}"
)

if not unresolved_components.empty:
    print(
        "\nComponents requiring interpretation from the displayed "
        "column inventories:"
    )

    display(
        unresolved_components
    )

## 1.14. Save a machine-readable preliminary reconstruction contract

The contract records exact paths, known windows, all candidate date fields, and the unresolved components. It is designed to be consumed by the later pre-baseline-only reconstruction notebook after the ambiguous fields have been resolved.

In [ ]:
# ============================================================
# 14. Preliminary reconstruction contract
# ============================================================

preliminary_contract = {
    "project_root": str(
        PROJECT_ROOT
    ),

    "authoritative_cohort_path": str(
        COHORT_PATH
    ),

    "mci_trajectory_path": str(
        MCI_TRAJECTORY_PATH
    ),

    "baseline_aligned_manifest_paths": {
        modality: str(
            path
        )
        for modality, path in MANIFEST_PATHS.items()
    },

    "cleaned_source_paths": {
        modality: str(
            path
        )
        for modality, path in SOURCE_PATHS.items()
    },

    "symmetric_windows_days": (
        SYMMETRIC_WINDOWS_DAYS
    ),

    "cohort_baseline_date_candidates": (
        cohort_baseline_candidates
    ),

    "trajectory_baseline_date_candidates": (
        trajectory_baseline_candidates
    ),

    "cohort_conversion_date_candidates": (
        cohort_conversion_candidates
    ),

    "trajectory_conversion_date_candidates": (
        trajectory_conversion_candidates
    ),

    "modality_date_candidates": (
        modality_date_candidates.to_dict(
            orient="records"
        )
    ),

    "unresolved_components": (
        unresolved_components[
            [
                "COMPONENT",
                "CANDIDATES",
                "NOTES",
            ]
        ].to_dict(
            orient="records"
        )
    ),
}


PRELIMINARY_CONTRACT_PATH = (
    CONTRACT_DIR
    / "preliminary_temporal_reconstruction_contract.json"
)


with PRELIMINARY_CONTRACT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        preliminary_contract,
        file,
        indent=2,
    )


print("=" * 72)
print("PREREQUISITES AUDIT COMPLETE")
print("=" * 72)

print(
    f"\nPreliminary contract saved to:\n"
    f"{PRELIMINARY_CONTRACT_PATH}"
)

print(
    f"\nAll audit tables saved under:\n"
    f"{TABLE_DIR}"
)

print(
    "\nThe next notebook should not be written until the "
    "reconstruction-readiness table has been reviewed and every "
    "required date field and tie rule has been fixed explicitly."
)